# 00 · Setup (Colab A100)

Chạy **một lần mỗi session** trước khi chạy model.
Dữ liệu RIÊNG TƯ không nằm trong repo — upload lên Drive `MyDrive/asr_benchmark_data/`:
`Auto_AutoELE/`, `SOL_UPS/`, `Data Audio_Auto_AutoELE_Grid.csv`, `Data Audio_SOL_UPS_Grid.csv`.


In [ ]:
!nvidia-smi -L

In [ ]:
![ -d benchmark-for-asr-model ] || git clone https://github.com/jajqja/benchmark-for-asr-model.git
%cd benchmark-for-asr-model
!git pull -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE_DATA = '/content/drive/MyDrive/asr_benchmark_data'  # sửa nếu để nơi khác
assert os.path.isdir(DRIVE_DATA), f'Upload data lên {DRIVE_DATA} trước'
for name in ['Auto_AutoELE','SOL_UPS','Data Audio_Auto_AutoELE_Grid.csv','Data Audio_SOL_UPS_Grid.csv']:
    src=os.path.join(DRIVE_DATA,name)
    if os.path.exists(src) and not os.path.exists(name): os.symlink(src,name)
    print(('OK  ' if os.path.exists(name) else 'MISS')+'  '+name)

In [ ]:
# Kết quả lưu ra Drive -> resume được khi session chết
RESULTS_DRIVE='/content/drive/MyDrive/asr_benchmark_results'
os.makedirs(RESULTS_DRIVE, exist_ok=True)
!rm -rf results && ln -s $RESULTS_DRIVE results
!mkdir -p results/hypotheses results/metrics results/reports
print('results ->', os.path.realpath('results'))

In [ ]:
!pip install -q -r requirements/base.txt
!python scripts/prepare_manifest.py --config configs/datasets/auto_autoele.yaml
!python scripts/prepare_manifest.py --config configs/datasets/sol_ups.yaml

### Subset ~200 utt (giữ tỉ lệ domain) — benchmark nhanh (~10h cho cả 13 variant)
Đổi `--n` nếu muốn nhiều/ít hơn. Đây là manifest mặc định trong `01_run_model`.


In [ ]:
!python scripts/subset_manifest.py \
  --in data/manifests/auto_autoele.jsonl \
  --out data/manifests/auto_autoele_sub200.jsonl --n 200 --seed 42
!wc -l data/manifests/*.jsonl